In [1]:
# Импорт класса SparkSession для работы с DataFrame и SQL API
from pyspark.sql import SparkSession

# Создание или подключение к существующей SparkSession
spark = (
    SparkSession.builder
        .appName("L1_Apache_Spark")              # Имя приложения в интерфейсе Spark
        .master("local[4]")                      # Локальный режим исполнения на 4 ядрах
        .config("spark.executor.memory", "2g")   # Объём памяти на каждого исполнителя
        .config("spark.driver.memory", "2g")     # Объём памяти, выделяемый драйверу
        .config("spark.python.worker.timeout", "12000")  # Таймаут для Python-воркеров (в секундах)
        .getOrCreate()                           # Возвращает существующую сессию или создаёт новую
)

# Извлечение SparkContext для работы с RDD и установки уровня логирования
sc = spark.sparkContext
sc.setLogLevel("WARN")  # Уровень WARN фильтрует информационные и отладочные сообщения


Загрузка данных

In [10]:
from dataclasses import dataclass
from datetime import datetime
import re
import xml.etree.ElementTree as ET
from typing import Iterator, List
from pyspark import SparkContext, RDD
from itertools import islice

In [7]:
# Регэксп один раз на уровне модуля: ищем именно теги в формате <tag1><tag2>…
_TAG_PATTERN = re.compile(r"<([^>]+)>")

# Выбор строк, которые точно содержат полный XML‑тег <row … />
_ROW_LINE_PATTERN = re.compile(r"^\s*<row\b.*?/\s*>\s*$")

@dataclass(frozen=True)
class Post:
    """Immutable Data Transfer Object для одного поста."""
    creation_date: datetime
    tags: List[str]

def load_and_filter_xml_lines(path: str) -> RDD[str]:
    """
    1. Считываем весь файл построчно.
    2. Оставляем только строки, представляющие полноценные <row ... />.
    """
    return (
        sc.textFile(path)
          .filter(lambda line: bool(_ROW_LINE_PATTERN.match(line)))
    )

def parse_rows_to_posts(lines_rdd: RDD[str]) -> RDD[Post]:
    """
    Преобразуем RDD строк с <row ... /> в RDD объектов Post:
      - в mapPartitions находим все исправно распарсенные элементы
      - фильтруем по корректности даты и тегов
    """
    def _partition_parser(lines: Iterator[str]) -> Iterator[Post]:
        for line in lines:
            try:
                elem = ET.fromstring(line)
                # вытаскиваем и разбираем теги из атрибута Tags
                raw_tags = elem.get("Tags", "")
                tags = _TAG_PATTERN.findall(raw_tags) if raw_tags else []

                # парсим дату, строго по формату ISO с микросекундами
                created_str = elem.get("CreationDate", "")
                created_dt = datetime.strptime(created_str, "%Y-%m-%dT%H:%M:%S.%f")

                yield Post(creation_date=created_dt, tags=tags)

            except ET.ParseError:
                # не XML или обрезанная строка — пропускаем
                continue
            except (TypeError, ValueError):
                # отсутствие даты или неверный формат — пропускаем
                continue

    return lines_rdd.mapPartitions(_partition_parser)

# === Пример «скелета» основного потока ===
xml_lines_rdd = load_and_filter_xml_lines("/content/posts_sample.xml")
posts_rdd     = parse_rows_to_posts(xml_lines_rdd)

# Возьмём 10 случайных постов без повторений
sample_posts = posts_rdd.takeSample(withReplacement=False, num=10)

for post in sample_posts:
    print(f"{post.creation_date.isoformat()} — теги: {post.tags}")

2014-07-24T22:10:11.633000 — теги: ['c#', 'winforms', 'listview', 'listviewgroup']
2015-11-22T19:29:56.053000 — теги: ['php', 'angularjs', 'rest', 'laravel']
2019-04-12T10:02:38.537000 — теги: ['mysql', 'mysql-5.6']
2017-02-09T14:22:44.353000 — теги: ['c#', 'json', 'rest', 'webinvoke', 'webget']
2015-06-02T17:53:13.193000 — теги: ['android', 'builder']
2016-03-01T15:39:24.963000 — теги: []
2010-03-05T16:13:34.037000 — теги: []
2017-04-08T03:32:34.690000 — теги: []
2018-02-17T08:41:57.717000 — теги: []
2013-03-21T13:07:19.340000 — теги: []


In [12]:
def _skip_header(partition_idx: int, rows: Iterator[str]) -> Iterator[str]:
    """
    Пропускает первую строку (заголовок) только в нулевой партиции.
    Благодаря этому не нужно делать .first() + .filter(),
    и весь файл читается за один проход.
    """
    if partition_idx == 0:
        # пропустить первую строку в первой партиции
        return islice(rows, 1, None)
    return rows

def load_programming_languages(csv_path: str) -> List[str]:
    """
    Загружает список языков программирования из CSV-файла.

    Шаги:
      1. Считываем файл как текст.
      2. mapPartitionsWithIndex(_skip_header): убираем единственный заголовок.
      3. Разбиваем строку по первой запятой и берём первый столбец.
      4. Приводим к нижнему регистру.
      5. collect(): собираем в список (подходит для небольших данных).
    """
    return (
        sc
        .textFile(csv_path)
        .mapPartitionsWithIndex(_skip_header)
        .map(lambda row: row.split(",", 1)[0].lower())
        .collect()
    )

# Загружаем данные
programming_languages = load_programming_languages("/content/programming-languages.csv")

# Покажем первые 10 элементов
programming_languages[:10]

['a# .net',
 'a# (axiom)',
 'a-0 system',
 'a+',
 'a++',
 'abap',
 'abc',
 'abc algol',
 'abset',
 'absys']

Объединение постов по годам

In [14]:
# Создаём RDD пар (год, пост)
postsByYear = posts_rdd.keyBy(lambda post: post.creationDate.year)


Подсчет количества постов для каждого года для каждого языка

In [21]:
from collections import Counter

langs_set = set(programming_languages)

counts_by_year = (
    posts_rdd
    .map(lambda p: (
        p.creation_date.year,
        Counter(t for t in p.tags if t in langs_set)
    ))
    .reduceByKey(lambda a, b: a + b)
)

for year, counts in sorted(counts_by_year.collect()):
    print(year, dict(counts))


2008 {'javascript': 2, 'java': 5, 'groovy': 1, 'io': 1, 'php': 1, 'python': 1, 'x++': 1, 'ruby': 4, 'c': 2}
2009 {'ruby': 8, 'xpath': 2, 'delphi': 7, 'haskell': 4, 'bash': 3, 'zsh': 1, 'php': 22, 'c': 6, 'objective-c': 6, 'java': 28, 'python': 23, 'javascript': 12, 'perl': 1, 'f#': 1, 'id': 1, 'prolog': 1, 'rpg': 1, 'matlab': 1, 'actionscript': 2, 'powershell': 1, 'scala': 1}
2010 {'java': 52, 'php': 46, 'ruby': 12, 'c': 20, 'python': 26, 'javascript': 44, 'applescript': 3, 'sed': 2, 'objective-c': 23, 'r': 3, 'matlab': 1, 'bash': 3, 'basic': 1, 'go': 1, 'delphi': 8, 'perl': 3, 'f#': 2, 'powershell': 1, 'ocaml': 1, 'mouse': 1, 'scheme': 1, 'racket': 1, 'haskell': 2, 'xpath': 1, 'actionscript': 1, 'dbase': 1, 'scala': 1, 'ksh': 1}
2011 {'c': 24, 'objective-c': 34, 'javascript': 83, 'java': 93, 'php': 102, 'ruby': 20, 'coldfusion': 4, 'perl': 9, 'bash': 7, 'clojure': 2, 'actionscript': 4, 'python': 37, 'delphi': 8, 'scheme': 1, 'glsl': 1, 'r': 3, 'jython': 1, 'matlab': 5, 'abap': 1, 'cyt

Создание сводки топ-10 по годам

In [25]:
from heapq import nlargest

# Для каждого года берём топ-10 языков по количеству
programming_languages_top = (
    counts_by_year
    .mapValues(lambda counts: [
        lang for lang, _ in nlargest(10, counts.items(), key=lambda kv: kv[1])
    ])
    .sortByKey()
)

for year, top_langs in programming_languages_top.take(14):
    print(year, top_langs)


2008 ['java', 'ruby', 'javascript', 'c', 'groovy', 'io', 'php', 'python', 'x++']
2009 ['java', 'python', 'php', 'javascript', 'ruby', 'delphi', 'c', 'objective-c', 'haskell', 'bash']
2010 ['java', 'php', 'javascript', 'python', 'objective-c', 'c', 'ruby', 'delphi', 'applescript', 'r']
2011 ['php', 'java', 'javascript', 'python', 'objective-c', 'c', 'ruby', 'perl', 'delphi', 'bash']
2012 ['php', 'javascript', 'java', 'python', 'objective-c', 'ruby', 'c', 'bash', 'r', 'matlab']
2013 ['javascript', 'php', 'java', 'python', 'objective-c', 'c', 'ruby', 'r', 'bash', 'scala']
2014 ['javascript', 'java', 'php', 'python', 'objective-c', 'c', 'r', 'ruby', 'bash', 'matlab']
2015 ['javascript', 'java', 'php', 'python', 'r', 'c', 'objective-c', 'ruby', 'matlab', 'scala']
2016 ['javascript', 'java', 'php', 'python', 'r', 'c', 'ruby', 'bash', 'scala', 'matlab']
2017 ['javascript', 'java', 'python', 'php', 'r', 'c', 'typescript', 'objective-c', 'ruby', 'powershell']
2018 ['python', 'javascript', 'java

Создание файла с отчетом

In [26]:
from pyspark.sql import SparkSession

# Инициализируем SparkSession
spark = (
    SparkSession
    .builder
    .appName("L2_Apache_Spark")
    .getOrCreate()
)

# Превращаем RDD в DataFrame с колонками "year" и "top_languages"
lang_top_df = (
    programming_languages_top
    .toDF(["year", "top_languages"])
)

# Записываем результат в Parquet (перезаписываем при наличии старых данных)
lang_top_df.write.mode("overwrite").parquet("top_10_languages_by_year.parquet")

# Показываем всё содержимое без обрезки
lang_top_df.show(truncate=False)


+----+--------------------------------------------------------------------------------+
|year|top_languages                                                                   |
+----+--------------------------------------------------------------------------------+
|2008|[java, ruby, javascript, c, groovy, io, php, python, x++]                       |
|2009|[java, python, php, javascript, ruby, delphi, c, objective-c, haskell, bash]    |
|2010|[java, php, javascript, python, objective-c, c, ruby, delphi, applescript, r]   |
|2011|[php, java, javascript, python, objective-c, c, ruby, perl, delphi, bash]       |
|2012|[php, javascript, java, python, objective-c, ruby, c, bash, r, matlab]          |
|2013|[javascript, php, java, python, objective-c, c, ruby, r, bash, scala]           |
|2014|[javascript, java, php, python, objective-c, c, r, ruby, bash, matlab]          |
|2015|[javascript, java, php, python, r, c, objective-c, ruby, matlab, scala]         |
|2016|[javascript, java, php, py

In [28]:
sc.stop()